# `C12` — Settlement and the cash leg

Scenario **C12** of the Track A certification ladder (card `OMIE-CERT-C12-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C12). Oracle kind: **exact-parametric, carried by I6**. **High-risk scenario** (independence rule 4): the K4 audit is a genuinely separate agent.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion on the trades and on the ledger is an **exact integer equality** on order ids, agent ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and trade values in **tick·tick units** ($D = q\,(p_{int} - O)$, see section 5). The engine reports its settlements as floats in EUR, so those — and only those — are compared **within a tolerance derived from the arithmetic** (the card's I6 restatement); no euro figure is ever compared exactly.

## 1. Purpose

**Purpose.** Everything above certifies that the right trades happen; this certifies that the right
**money** changes hands. OMIE values a trade as quantity × price × duration, with duration fixed at a
quarter hour — the $\times d$ factor is a genuine departure from equity-market convention and one of the
assumption violations §10 tabulates. The strongest available check is that the cash leg sums to zero
across participants. Two halves make it non-trivial: each trade's $Q\cdot p\cdot d$ is recomputed
exactly in ticks against its row, and the engine's own per-participant settlements must sum to zero
within the tolerance derived from the settlement arithmetic. (Summing the rows in integers would be zero by
algebra — a ledger checked against itself, which rule 1 forbids; and the engine reports its settlement
as a float, so no integer settlement can be read from it.) The euro figure is
computed and reported but never asserted — the price decode is FMA-contracted and backend-unstable,
so a euro assertion would be flaky between CPU and GPU while proving nothing extra.

**Two claims deliberately narrowed.** §8.3's per-unit identity `agent_pnl = Σ unit_pnl` is
**untestable today**: `OMIEOrder` has no unit field and the settlement function is agent-level,
because the loader folds `(Agente, Unidad)` into a single identifier — so a simulator "agent" already
*is* a unit. And "zero-sum" holds of the **cash leg** only: §8.4 defines PnL as cash *plus* an
explicitly exogenous energy-value term, so economic zero-sum is not a claim this simulator can make.

**What this scenario cannot catch: a trade at the wrong price.** The engine computes settlement *from
its own trade rows*; there is no second, independent valuation anywhere in the system. If a trade were
booked at the wrong price, the trade row and the settlement would both carry that wrong price and
agree perfectly. So C12 certifies the **arithmetic** of settlement — $Q\cdot p\cdot d$, the sign convention, the
zero-sum cash leg, and that the reported settlement agrees with the rows written — and says plainly
that **mispricing is caught elsewhere**: by C2 (the resting-price rule, as an exact formula) and by
invariant I5 (every trade at its price-setter's own limit) on every scenario.

**A known hazard C12 must not fall into.** `compute_agent_pnl` expects a flat list of trades, shaped
(M, 6), but does not check it. Given the environment's per-step trade tape, shaped (steps, 2, 6), it
silently returns a wrong value — observed 0.0 (filed 2026-10-01, unscheduled, as
`OMIE-AGENT-PNL-SHAPE-GUARD-1`). A zero for every participant sums to zero, so the cash-leg check
would pass on nothing. C12 therefore always flattens the trades to (M, 6) before settling, as the
production callers do, asserts the shape it passes, and guards against a vacuous zero-sum in two
ways: the gross cash leg $\sum \lvert V\rvert$ over all trades must be positive (I9), and each
participant's reported settlement must equal an independent per-row recomputation of
$\sum (\pm Q\cdot p\cdot d)$ for that participant — which a mis-shaped call fails even when its output
is 0. (A per-participant "non-zero" check would be wrong: a legal self-match or an equal-price round
trip settles to exactly zero — domain review v3.0, V3-03.)

*(Verbatim from `scenario-ladder.md` §C12.)*

**Certifies** (card): §8.1 $V = Q\cdot p\cdot d$ with $d = 0.25\ \text{h}$; §8.4's sign convention; §11.1 row *"Trade value = Q × p × d"*; OQ-18 for the x64 scoping. **Pass** (card): I6(a) and (b) as restated in the battery; per-trade $Q\cdot p\cdot d$ recomputed exactly in ticks against each row; euro figures otherwise reported, not asserted; the eager and jitted runs agree; the full battery runs; I8 capacity headroom; I9 non-vacuity (the gross cash leg $\sum\lvert V\rvert > 0$); every quote matches its source byte for byte. **Does not certify:** mispricing (C02 and I5 own it); §8.3 per-unit granularity (not implemented: a simulator agent already is a unit); §8.2 trade firmness (not represented); economic zero-sum (§8.4's energy-value term is exogenous). **Failure implicates:** settlement arithmetic, the $d$ factor, the sign convention, or a report/tape disagreement.

**The construction** (built in section 4 from a seeded PCG64 stream; there is no generator registry key for C12). Each draw is one session on one contract, one fresh book, NON and IOC orders only:

| phase | rows | what is drawn |
|---|---|---|
| price regime | — | one of six: *ordinary* (mid 5–300 EUR/MWh), *negative* (mid −500 to −5), *zero* (mid exactly 0.00, narrow), *large_qty* (ordinary prices, 220–999 MW orders), *extreme_high* (mid 5 000–9 900, 440–999 MW), *extreme_low* (mid −9 900 to −5 000, 440–999 MW) |
| resting book | 1–10 NON orders per side, every bid below every ask (in the *zero* regime one of them sits exactly at 0.00 EUR/MWh) | $A \in [2, 5]$ participants (agents $1..A$), drawn per order |
| order flow | 2–20 orders, NON or IOC; the first two are one BUY priced through the whole ask side and one SELL through the whole bid side (so **both sides trade**), the rest anywhere within the regime's half-width | agents $1..A$: self-matches occur naturally (§6.6) |
| self-match pair (30 % of draws, when the book has a gap) | agent 9 rests a NON at the best price inside the spread, then sends an IOC against it at the same price | a participant whose cash leg is **exactly zero** (review V3-03) |

**What is asserted, per valid draw** (two assertions, expected and observed recorded **in full** as canonical compact JSON): **(1) integer** — every trade row `[buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]` (D3-08: the counterparty sequence is pinned), each trade's value $D_i = q_i\,(p_i - O)$ recomputed from its own row, each submission's traded flag, reported quantity and $D_s$, each participant's ledger `[bought, sold, D_a]`, the gross $\sum_i\lvert D_i\rvert$, the set of participants the engine settled, the shape `(M, 6)` of the settlement tape, and the per-order fates (filled, still resting, IOC remainder cancelled); **(2) euro, within the derived tolerance** — each participant's engine settlement $S_a$ (as recorded by the record builder **and** as returned by this notebook's own flattened call) against $k\,D_a$, and each trading submission's reported settlement against $k\,D_s$. Then I6 (a) and (b) and the rest of the battery on every draw, the discrimination, coverage and capacity checks and the eager comparisons (section 7).

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.13** (the v2.13 change touched §4.2.1 Rule 3 and §6.2 item 7, OQ-31; neither is used here). Every quote is checked against the current file at run time.
- **F2 — the dataset is built in the notebook.** The generator has no C12 mode. `C15.random_session` and `C06.multi_level_walk` were considered and not reused: C12 needs price regimes those constructions do not draw (negative prices, a trade at exactly 0.00, and the envelope extremes where OQ-18's integer products leave int32), a dedicated self-match pair, and a session whose every trade follows from §6.1 and §6.2 items 1–6 alone, so that the reference walk (W) is exact without iceberg or fill-or-kill semantics. Section 4 builds every tape with the generator's public helpers (`TapeBuilder`, `tape_capacity_bounds`, `check_envelope`, `Draw`) from `numpy.random.Generator(PCG64(seed))` and records every drawn value in `draw.params`. A seed whose analytic bounds breach the envelope (`CapacityBreach`), or none of whose 64 attempts passes the acceptance test of section 4, is refused and counted as a discard. The results JSON lists no generator digest (`constructions=()`); section 4 prints the construction's digest over seeds 0–7 with the `construction_digest` formula. `draw.expected` is empty and never read.
- **F3 — the expectation uses the specification's constants, not the configuration's.** $O = 999\,900$ (−9 999 EUR/MWh at 0.01 EUR/MWh, §2.3), $k = 0.1 \times 0.01 \times 0.25 = 2.5\times10^{-4}$ EUR per tick·tick (§2.3 ticks, §8.1 $d$). The battery's I6 takes $k$ from `OMIEProductConfig`, so a wrong $d$ in the configuration would move the battery's $V_s$ and the engine together; C12's own reconciliation would not, and section 7 also asserts that `OMIEProductConfig()` carries the specification's ticks, price floor and $d$.
- **F4 — the per-participant tolerance is C12's own derivation.** V10 pins $\tau_s$ per submission (I6(b)) and $\sum_s\tau_s$ for the sum over participants (I6(a)); it pins no tolerance for one participant's settlement against its rows. C12 uses, over participant $a$'s $m_a$ legs (a self-match row is two legs), $\tau_a = k\,u\,\big(m_a (W_a + O\,Q_a) + 5\sum_i\lvert D_i\rvert\big)(1 + (m_a + 8)u)$, with $u = 2^{-24}$, $W_a = \sum p_i q_i$, $Q_a = \sum q_i$, all over the legs, unsigned. It bounds **any** float32-or-better evaluation of $k\sum(\pm q_i(p_i - O))$ that rounds each row's two products, the subtraction and the scalings once each and accumulates the $m_a$ signed terms in float32 — so it depends on no implementation detail of `settlement.py`. It is $\ge$ V10's chunk-agnostic bracket over the same rows, and $\ge 5u\lvert k D_a\rvert$, so it also covers the domain review's statement (§16.1) that the per-agent error is at most $u\lvert\mathit{pnl}_a\rvert$. The observed error over $\tau_a$ is reported.
- **F5 — the per-submission reconciliation uses $c_s = m_s$.** The number of settlement chunks $c_s$ is read by the battery from engine timestamps; the expectation cannot know it, so C12's own check uses the chunk-agnostic $\tau_s$ (`i6_tolerance(..., chunks=None)`, recomputed with the specification's constants), which is looser. The battery's I6(b), with the observed $c_s$, also runs on every draw and must pass.
- **F6 — no integer zero-sum is asserted.** $\sum_a D_a = 0$ holds by algebra over any set of rows (review V3-04; rule 1 forbids checking a ledger against itself); it is shown in section 5, never checked. The non-trivial halves are asserted: each $S_a$ against its own per-row recomputation, and I6(a) on the engine's floats. No participant is required to have a non-zero settlement (V3-03): the self-match pair makes one participant's cash leg exactly zero on about a third of the draws, and that must reconcile like any other.
- **F7 — OQ-18's x64 scoping.** OQ-18 dropped the `jax.jit` contract of `compute_agent_pnl` and keeps an **eager** contract whose own scoped `enable_x64` carries the integer products past int32. C12 certifies the eager path at the envelope: coverage classes require trade rows with $q\,p_{int} \ge 2^{31}$ and with $\lvert q(p - O)\rvert \ge 2^{31}$, and a participant with $\lvert D_a\rvert \ge 2^{31}$; the settlement is called plainly eager, as production callers do (no outer `enable_x64`, no `jax.jit`). A jitted settlement is not certified (OQ-18: deferred).
- **F8 — the shape hazard (`OMIE-AGENT-PNL-SHAPE-GUARD-1`).** Section 6's `settle` flattens the fills to an `(M, 6)` int32 tape before calling `compute_agent_pnl`, and section 7 asserts the shape it passed on every draw. A mis-shaped call (observed: 0.0 for everyone) is the "zero" wrong reading below, which every valid draw discriminates.
- **F9 — which battery checks must pass, and which may be not applicable.** Every valid draw has at least two trades by construction. RB, I2, I3, **I6**, I7 and I9 apply to every draw and must **pass**. I1, I4 and I5 are `not_applicable` only on a run with no fills, which the oracle already fails; for them the assertion is **"no draw returns fail"**, and the not-applicable counts are reported separately (the C10 K6-F lesson). I8 `invalid` discards the draw.
- **F10 — a capacity flag inside the margin is a failure, not only a discard** (as C08–C11).
- **F11 — the draw envelope is empirical.** Mid prices, half-widths (1–60 ticks; 1–5 in the *zero* regime), book sizes, quantities (0.1–20 MW ordinary; 220–999 MW and 440–999 MW in the large regimes) and the participant count are test ranges, never bounds; every price and quantity stays inside §2.3.
- **F12 — discrimination.** From the tape alone, section 5 evaluates named wrong settlements and section 7 asserts that each of the first four differs from the rule by more than $5\tau_a$ for at least one participant on **every** valid draw: **zero** (a vacuous or mis-shaped call), **sign flipped**, **$d = 1$ h**, **price not offset-decoded**. Three int32 wrap readings (OQ-18's overflow) are discriminated only on envelope draws; each must be discriminated on at least one valid draw (coverage).
- **F13 — not drawn:** icebergs, fill-or-kill, cancellations, gate expiry, more than one contract, batch matching (§8.1's *"arithmetic mean of last pair in batch matching"* is §6.3, multi-zone only). Their settlement is reached by I6 in C04, C08–C11 and C15; recorded as coverage limits, not claims.
- **F14 — euro figures.** Apart from the two tolerance comparisons the card's pass criterion names, every euro figure (the gross cash leg, each $S_a$, the error ratios) is reported only.
- **F15 — eager and jitted.** `compare_records` compares the integer fields only (V2: float fields legitimately differ between modes). Each eager record's settlements and reports are also reconciled within the derived tolerance.
- **F16 — §8.4's mark-to-market term** (*"plus mark-to-market of open positions if applicable"*) is not applicable to the cash leg and not claimed; the residual NON orders resting at the end carry no cash.


## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **The value of a trade (§8.1; §2.3; §10; §11.1).** $V = Q \cdot p \cdot d$ with $d = 0.25$ h, $Q$ in MW and $p$ in EUR/MWh; with §2.3's ticks and price floor this is exactly $k\,q\,(p_{int} - O)$ in integer ticks. §10 is why the $\times d$ factor is the departure most likely to be lost.
- **Who pays whom (§8.4).** The buyer's cash leg is $-V$, the seller's $+V$, a participant's net is $\sum_{\text{sells}} V - \sum_{\text{buys}} V$. The energy-value terms are **exogenous**, which bounds every zero-sum claim to the cash leg.
- **Self-matching is legal (§6.6; review V3-03).** A row whose buyer and seller are the same participant settles to zero for that participant, and so can an equal-price round trip; no participant is required to have a non-zero settlement.
- **What is and is not a check (review V3-03, V3-04).** Each participant's reported settlement against an independent per-row recomputation; the integer sum over participants is algebra, not a check.
- **OQ-18.** The int32 range is exceeded by the per-row products at the envelope, and the eager settlement path carries its own x64 scope.
- **The trades themselves (§6.1; §6.2 items 1–6; §4.5; §5.1; review D3-08).** The reference walk that pins every row, counterparties included.
- **What is not certified (§8.3).** The per-unit identity is a hard rule the simulator cannot express.


In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- the value of a trade ------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§8.1',
        'V (EUR) = Q (MW) × p (EUR/MWh) × d (h)',
        note='The settlement formula: V = Q x p x d.',
    ),
    SpecQuote(
        'market-spec.md',
        '§8.1',
        '`V`: financial value in EUR',
        note="V is in EUR: the unit of the engine's settlements.",
    ),
    SpecQuote(
        'market-spec.md',
        '§8.1',
        '`Q`: traded quantity in MW',
        note='Q in MW: q_int x 0.1 (§2.3).',
    ),
    SpecQuote(
        'market-spec.md',
        '§8.1',
        "`p`: trade price in EUR/MWh (resting order's limit price in regular matching; arithmetic mean of last pair in batch matching)",
        note="p is the trade's own price, the resting order's limit in regular matching: C12 values each row at the price it carries (mispricing is C02's and I5's).",
    ),
    SpecQuote(
        'market-spec.md',
        '§8.1',
        '`d`: delivery duration of the contract in hours — **for 15-min contracts: d = 0.25**',
        note='d = 0.25 h for the quarter-hour contracts the simulator trades: k = 0.1 x 0.01 x 0.25 EUR per tick-tick (finding F3).',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Trade value formula:** `V (EUR) = Q (MW) × p (EUR/MWh) × d (h)` where d = 0.25 h for 15-min contracts. Source: Continuous Trading Matching Algorithm §5.3.',
        note="The same formula, stated with the product's units.",
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Price bounds:** −9,999 EUR/MWh (minimum) to +9,999 EUR/MWh (maximum).',
        note='The price floor -9,999 EUR/MWh is the offset O = 999,900 of the integer price encoding; every drawn price stays inside the bounds.',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Price tick (minimum increment):** 0.01 EUR/MWh (2 decimal places maximum).',
        note='The price tick: p = (p_int - O) x 0.01 EUR/MWh.',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Quantity bounds:** 0.1 MW (minimum) to 999 MW (maximum).',
        note='The quantity bounds: every drawn quantity is 1..9,990 ticks.',
    ),
    SpecQuote(
        'market-spec.md',
        '§2.3',
        '**Quantity tick (minimum increment):** 0.1 MW (1 decimal place maximum).',
        note='The quantity tick: Q = q_int x 0.1 MW.',
    ),
    SpecQuote(
        'market-spec.md',
        '§10',
        '| Trade value = Q × p | Trade value = Q × p × d (delivery duration factor; d = 0.25 h for 15-min) | Financial formula differs |',
        note="The x d factor is a departure from the equity convention: the 'd = 1 h' wrong reading of section 5 is what forgetting it looks like.",
    ),
    SpecQuote(
        'market-spec.md',
        '§11.1',
        '| Trade value = Q × p × d; d = 0.25 h for 15-min contracts | Continuous Trading Matching Algorithm §5.3 |',
        note='The hard-rule row the card certifies.',
    ),
    # --- who pays whom: the cash leg --------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§8.4',
        '**Buy agent PnL per trade:** `−V` (cash outflow) + market value of acquired energy (exogenous to the simulator; relevant for imbalance resolution).',
        note="The buyer's cash leg is -V; the energy value is exogenous (finding F16: cash leg only).",
    ),
    SpecQuote(
        'market-spec.md',
        '§8.4',
        '**Sell agent PnL per trade:** `+V` (cash inflow) − cost of delivered energy (exogenous).',
        note="The seller's cash leg is +V; the cost of delivered energy is exogenous.",
    ),
    SpecQuote(
        'market-spec.md',
        '§8.4',
        '**Net realized PnL:** `sum(sell_trades: V) − sum(buy_trades: V)`, plus mark-to-market of open positions if applicable.',
        note="A participant's net cash: sells minus buys. This is S_a, the quantity each participant's engine settlement must reconcile to.",
    ),
    # --- self-matching, and what is (not) a check ---------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§6.6',
        "Matching is pure price-time priority; an agent's buy order can match its own sell order, and orders from two different units of the same agent can match against each other.",
        note='Self-matches are legal: a self-match row is one leg of each sign for the same participant, so it nets to zero.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.6',
        '**Simulator implication:** The matching engine must not reject or skip matches based on order ownership.',
        note='The construction draws same-agent pairs on purpose (the self-match pair, and agents 1..A on both sides).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-03',
        "and each participant's reported settlement must\nequal an independent per-row recomputation of $\\sum (\\pm Q\\cdot p\\cdot d)$ for that participant — which\na mis-shaped call fails even when its output is 0.",
        note='The vacuity guard C12 asserts: the gross cash leg is positive and each participant reconciles to its own rows.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-03',
        'A participant who buys and sells the same quantity at the same price\n  has exactly zero cash',
        note='Why no participant must have a non-zero settlement.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'V3-04',
        'If the integer sum is computed from trade rows, it is zero by algebra: each row is $+V$ to its seller\nand $-V$ to its buyer.',
        note='Why the integer sum over participants is shown, never asserted (finding F6).',
    ),
    # --- OQ-18: the x64 scoping ---------------------------------------------------------------------------
    SpecQuote(
        'open-questions.md',
        'OQ-18',
        "the x64 flag is read at trace time, so `int64` casts inside\nthe traced body silently collapse to `int32`, losing the two-term\naccumulator's overflow guarantees at the OMIE envelope (per-row\n`Q_tick × p_tick_offset ≈ 2×10¹⁰`, outside int32 range ~2.15×10⁹).",
        block='Context.',
        note='The integer products leave int32 at the OMIE envelope: the large and extreme regimes reach them (finding F7).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-18',
        'The function remains JAX-traceable and supports `jax.vmap` when the caller\nsupplies an outer `with jax.experimental.enable_x64():` context. The body\nopens its own scoped `enable_x64` for eager callers (a no-op under tracing,\nactive on the eager path).',
        block='Option 3a adjudication',
        note='The eager contract certified here: settle() calls the settlement eagerly, with no outer x64 context and no jit.',
    ),
    # --- the trades themselves: the reference walk --------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Buy side:** ranked highest price first; ties broken by earliest timestamp.',
        note='A SELL aggressor meets the highest bid first; equal prices by entry.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Sell side:** ranked lowest price first; ties broken by earliest timestamp.',
        note='A BUY aggressor meets the lowest ask first; equal prices by entry.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 1',
        'Incoming order attempts to match against best counterpart in the consolidated order book.',
        note='Every new row is an incoming order matched against the book.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note='The crossing condition of the walk.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Each trade row carries the resting order's price.",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 4',
        'If incoming order is partially filled and has NON restriction: residual enters the book with its original entry timestamp (see §5.2).',
        note='A NON residual rests, keeping its entry timestamp.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 5',
        'If incoming order is partially filled and has IOC restriction: residual is cancelled.',
        note='An IOC residual is cancelled (cause ioc_remainder in the record).',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 6',
        'If matching exhausts one price level, continues to next best price level.',
        note='An aggressor walks level after level: one submission can write several rows (m_s >= 2).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| NON | Yes | Yes | Regular, Block |',
        note='NON: partial fills, the remainder enters the book.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| IOC | Yes | No (cancelled) | Regular |',
        note='IOC: partial fills, never enters the book.',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.1',
        'CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)',
        note="The IOC remainder's fate: CANCELLED.",
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Good practice adopted here: every trade row is pinned by buyer and seller order id and agent.',
    ),
    # --- what is not certified ----------------------------------------------------------------------------
    SpecQuote(
        'market-spec.md',
        '§8.3',
        'The primary settlement granularity is the **bidding unit (Unidad)**, not the agent.',
        note='Not certified: the simulator folds (Agente, Unidad) into one identifier, so a simulator agent already is a unit.',
    ),
    SpecQuote(
        'market-spec.md',
        '§8.3',
        'This sum-over-units invariant must hold exactly.',
        note='The per-unit identity is untestable today (card: Does not certify).',
    ),
)
CITATIONS = (
    Citation(
        'Continuous Trading Matching Algorithm (public description) §5.3 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:576-588',
        'Whenever two orders are matched, a trade is created. Even though one order can match with multiple other or ders, a trade is always between exactly two orders. [...] The financial value of a trade is calculated according to the below formula: V (EUR) = Q (MW) * p (EUR/MWh) * d (h) Where: V is the value of the trade in Euro. Q is the quantity of the power traded in Megawatt. p is the price per energy unit agreed between buyer and seller in Euro per Megawatt hour. d is the duration of the delivery period of the traded contract in hours.',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")


## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws (the card's default for a random scenario), from seed `SEED0`. A seed names one draw.
- **Discards.** A seed the section-4 builder refuses (`CapacityBreach`, or no accepted attempt), a draw whose tape lies outside the capacity margin before the run, or a draw the battery's I8 marks `invalid` after it, is **discarded and replaced by the next seed**. The loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F10).
- **`EAGER_DRAWS = 2`.** The first `EAGER_DRAWS` **valid** draws also run eager, and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`), and reconciled within the derived tolerance. On top of these, an eager run is **forced** for the first valid draw of each eager class (the six price regimes and the self-match pair). A missing eager comparison for any eager class is a failed `eager_jit` assertion; eager runs are chosen among valid draws only.
- **The constants of the expectation are the specification's** (finding F3): $O$, $k$, $d$ below. `OMIEProductConfig()` is used only to build tapes and to be compared with them.
- **The classes** (each must appear among the valid draws: the coverage assertion of section 7) are listed by the next cell.


In [ ]:
CERT_ID = "C12"            # C00-C16 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws (card: random scenarios default to 500 draws)
SEED0 = 0                  # first seed
EAGER_DRAWS = 2            # the first EAGER_DRAWS valid draws also run eager (plus the first of each eager class)


In [ ]:
import pathlib
import types

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

CONSTRUCTION = "C12.cash_leg_session (built in section 4)"   # not a generator registry key (finding F2)
CONFIG = OMIEProductConfig()          # builds tapes; never a source of the expectation's constants
ENVELOPE = synth.DEFAULT_ENVELOPE
MAX_ATTEMPTS = 2 * DRAWS   # the draw loop stops at DRAWS valid draws or after 2 x DRAWS attempts

BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

# ---- the specification's constants (finding F3) ----------------------------------------------------
PRICE_TICK_EUR = 0.01                 # §2.3: EUR/MWh per price tick
QTY_TICK_MW = 0.1                     # §2.3: MW per quantity tick
D_HOURS = 0.25                        # §8.1: delivery duration of a 15-min contract, hours
PRICE_FLOOR_EUR = -9999               # §2.3: the price floor, the zero of the integer encoding
O_TICKS = 999_900                     # -PRICE_FLOOR_EUR / PRICE_TICK_EUR: p_int of 0.00 EUR/MWh
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW
K_EUR = QTY_TICK_MW * PRICE_TICK_EUR * D_HOURS   # EUR per (qty tick x price tick): V = K_EUR * q * (p_int - O)
U = 2.0 ** -24                        # float32 unit roundoff (V10)
INT32 = 2 ** 31                       # OQ-18: the int32 range the integer products leave
TAU_PIN = (1_004_598, 20, 6.043878e-4)   # V10's pinned example: one row p, q -> tau (EUR)

# ---- the draw envelope: empirical test ranges, never bounds (finding F11) -------------------------
# regime -> ((mid lo, mid hi) in price ticks relative to 0.00 EUR/MWh, (qty lo, qty hi) in qty ticks)
REGIMES = {
    "ordinary":     ((500, 30_000), (1, 200)),            # mid 5-300 EUR/MWh, 0.1-20 MW
    "negative":     ((-50_000, -500), (1, 200)),          # mid -500 to -5 EUR/MWh
    "zero":         ((0, 0), (1, 200)),                   # mid exactly 0.00 EUR/MWh
    "large_qty":    ((500, 30_000), (2_200, 9_990)),      # 220-999 MW: q * p_int >= 2^31
    "extreme_high": ((500_000, 990_000), (4_400, 9_990)),  # mid 5,000-9,900 EUR/MWh, 440-999 MW
    "extreme_low":  ((-990_000, -500_000), (4_400, 9_990)),  # mid -9,900 to -5,000 EUR/MWh
}
REGIME_NAMES = tuple(REGIMES)
REGIME_P = (0.30, 0.15, 0.15, 0.15, 0.125, 0.125)
HALF_WIDTH = (1, 60)                  # price ticks either side of the mid
HALF_WIDTH_ZERO = (1, 5)              # the zero regime: a narrow book around 0.00
BOOK_PER_SIDE = (1, 10)               # resting NON orders per side
FLOW = (2, 20)                        # order-flow rows (the first two: one per side, through the book)
P_IOC = 0.25                          # P(an order-flow row is IOC)
AGENTS = (2, 5)                       # participants 1..A
SELF_AGENT = 9                        # the self-match pair's participant
P_SELF_PAIR = 0.30                    # P(a self-match pair is appended, when the book has a gap)
MAX_TRIES = 64                        # attempts per seed before the seed is refused

# ---- the classes ------------------------------------------------------------------------------------
REGIME_CLASSES = tuple(f"regime: {r}" for r in REGIME_NAMES)
SELF_PAIR_CLASS = "a self-match pair (a participant whose cash leg is exactly zero)"
EAGER_CLASSES = REGIME_CLASSES + (SELF_PAIR_CLASS,)
TRADE_CLASSES = (
    "a trade at a negative price (V < 0)",
    "a trade at exactly 0.00 EUR/MWh (V = 0)",
    "a self-match trade row (buyer agent == seller agent)",
    "a participant who both bought and sold",
    "a submission writing >= 2 trade rows",
    "an IOC aggressor that traded",
    "a NON aggressor whose residual rested after trading",
    "OQ-18: a trade row with q * p_int >= 2^31",
    "OQ-18: a trade row with |q * (p_int - O)| >= 2^31",
    "OQ-18: a participant with |D_a| >= 2^31",
)
WRONG_ALWAYS = (
    "zero (a vacuous or mis-shaped settlement call)",
    "sign flipped (the buyer receives V, the seller pays it)",
    "d = 1 h in place of 0.25 h",
    "price not offset-decoded (p_int in place of p)",
)
WRONG_COVER = (
    "int32 wrap of each row's q * p_int",
    "int32 wrap of each row's two products q * p_int and q * O",
    "int32 wrap of the participant's accumulated D_a",
)
COVER_CLASSES = tuple(f"discriminated: {n}" for n in WRONG_COVER)
ALL_CLASSES = EAGER_CLASSES + TRADE_CLASSES + COVER_CLASSES

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C12_settlement_cash_leg.ipynb"

print(f"{CERT_ID}: {CONSTRUCTION}, {DRAWS} valid draws from seed {SEED0} (at most {MAX_ATTEMPTS} attempts)")
print(f"eager: the first {EAGER_DRAWS} valid draws + the first of each of {len(EAGER_CLASSES)} eager classes")
print(f"coverage classes ({len(ALL_CLASSES)}):")
for _c in ALL_CLASSES:
    print(f"  - {_c}")
print(f"spec constants: O = {O_TICKS} ticks, k = {K_EUR!r} EUR per tick-tick, d = {D_HOURS} h, u = 2^-24")
print(f"OMIEProductConfig(): price_tick {CONFIG.price_tick}, qty_tick {CONFIG.qty_tick}, "
      f"price_min {CONFIG.price_min}, delivery_duration_hours {CONFIG.delivery_duration_hours} (compared in section 7)")
print(f"capacity envelope: {ENVELOPE}")


## 4. Dataset

`make_draw(seed)` below builds one tape from `numpy.random.Generator(PCG64(seed))` with the generator's public helpers, and returns a generator `Draw` (finding F2). In order:

1. **The regime.** One of the six regimes of section 3, with probabilities 0.30 / 0.15 / 0.15 / 0.15 / 0.125 / 0.125, drawn once per seed.
2. **An attempt.** A mid price $m$ (exactly $O$, i.e. 0.00 EUR/MWh, in the *zero* regime), a half-width $H$ and a participant count $A$. **Resting book:** $R_B, R_S \in [1, 10]$ NON orders, bids at $m - \delta$ and asks at $m + \delta$, $\delta \in [1, H]$, so every bid is strictly below every ask; in the *zero* regime one of them is moved to exactly $m$. They are shuffled into one submission order. **Order flow:** $F \in [2, 20]$ orders, NON or IOC (probability 0.25); the first two are one BUY at $m + H$ and one SELL at $m - H$ in random order (each reaches through the whole opposite side, so both sides trade); the rest have a random side and a limit in $[m - H, m + H]$. Each order's quantity is drawn from the regime's range, its agent from $1..A$. **Self-match pair** (probability 0.3): the reference walk (W) is run on the rows so far; if the book leaves room, agent 9 rests a NON at the best price strictly inside the spread (one tick better than its side's best, never crossing), then sends an IOC of at most that size at the same price on the other side, which can meet only its own order.
3. **Rows.** `TapeBuilder(OMIEProductConfig())`: one contract (0), order ids $1..N$, tape timestamps $1, 2, \ldots$, every row GFS with `expiry_time = 0`, no peak, no increment, then pad rows.
4. **The acceptance test** (from (W) and the arithmetic of section 5, on the tape alone): a BUY aggressor and a SELL aggressor each traded; the gross cash leg is positive; each of the four always-discriminated wrong readings (finding F12) misses some participant by more than $5\tau_a$; the *zero* regime has a trade at exactly 0.00; the *large_qty* regime has a row with $q\,p_{int} \ge 2^{31}$; the *extreme* regimes a row with $\lvert q(p_{int} - O)\rvert \ge 2^{31}$; a self-match pair, when appended, traded only with itself. A rejected attempt is replaced by the next attempt **on the same random stream**, so a seed still names one draw; after 64 rejected attempts the seed is refused.

The analytic bounds (`tape_capacity_bounds`) are checked against the envelope by `check_envelope`, which refuses the seed (`CapacityBreach`) if any is exceeded. `draw.params` records every drawn value: `regime`, `mid_int`, `half_width_ticks`, `n_agents`, `attempt`, `rejected_attempts`, `self_pair` and every order (`oid, phase, side, exec, price_int, qty_int, agent`).

**The reference walk (W)** is defined in this section because the acceptance test uses it; section 5 states it as the known result. It is §6.1 and §6.2 items 1–6 with §4.5: an incoming order meets the opposite side ranked by price, then entry; while it has quantity left and the best opposite entry crosses ($\text{sell} \le \text{buy}$), it trades $\min(\text{left}, \text{resting})$ **at the resting entry's price**; a NON residual rests at its limit, an IOC residual is cancelled.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from the tape before the run: the draw's analytic bounds and the generic `tape_capacity_bounds(tape)` within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), and no iceberg row at all (so every iceberg is trivially within $\lceil \mathit{total}/\mathit{peak}\rceil \le 64$). A session has at most 42 rows, so every bound is far inside. After the run, the battery's I8 confirms the flags.

Below: the first draw in readable units beside the integers, its book shape and expected cash legs, the construction digest, and the envelope actually drawn in the initial seed range.


In [ ]:
import hashlib
import json
from collections import Counter

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_EXPIRY = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_VALIDITY,
                                               synth.COL_EXPIRY)
C_PEAK, C_PPD = synth.COL_PEAK, synth.COL_PPD
ORDER_KEYS = ("oid", "phase", "side", "exec", "price_int", "qty_int", "agent")


class ConstructionRefused(Exception):
    """No attempt of a seed passed the acceptance test (section 4, step 4)."""


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def eur(p_int):
    # Display only: the price in EUR/MWh (§2.3).
    return (int(p_int) - O_TICKS) * PRICE_TICK_EUR


def mw(q_int):
    # Display only: the quantity in MW (§2.3).
    return int(q_int) * QTY_TICK_MW


def crosses(rest_side, p_rest, p_in):
    # §6.2 item 2: sell price <= buy price, for a resting entry on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def wrap32(x):
    # Two's-complement int32 wrap of an integer (the OQ-18 overflow readings).
    return ((int(x) + 2 ** 31) % 2 ** 32) - 2 ** 31


# ---- the reference walk (W): §6.1, §6.2 items 1-6, §4.5 (NON and IOC rows only) -------------------------------------
def walk(tape):
    # Returns the trade rows [buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission],
    # the aggressor of each row, the final book, the IOC remainders and the NON aggressors that rested a residual.
    book, trades, aggressor, ioc_cancelled, rested = [], [], [], {}, []
    for s, r in enumerate(real_rows(tape)):
        side, p, q = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY])
        oid, ag, ex = int(r[C_OID]), int(r[C_AGENT]), int(r[C_EXEC])
        left = q
        while left > 0:
            opp = [e for e in book if e["side"] != side and crosses(e["side"], e["price"], p)]
            if not opp:
                break
            best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["entry"]))
            f = min(left, best["qty"])
            if side == BUY:
                trades.append([oid, best["oid"], ag, best["agent"], best["price"], f, s])
            else:
                trades.append([best["oid"], oid, best["agent"], ag, best["price"], f, s])
            aggressor.append({"oid": oid, "side": side, "exec": ex})      # the row prints at the resting price
            left -= f
            best["qty"] -= f
            if best["qty"] == 0:
                book.remove(best)
        if left > 0 and ex == NON:                                        # §6.2 item 4 / §4.5 NON
            book.append({"side": side, "price": p, "qty": left, "oid": oid, "agent": ag, "entry": s})
            if left < q:
                rested.append(oid)
        elif left > 0 and ex == IOC:                                      # §6.2 item 5 / §4.5 IOC
            ioc_cancelled[oid] = left
    return {"trades": trades, "aggressor": aggressor, "book": book, "ioc_cancelled": ioc_cancelled,
            "rested_after_trade": rested}


# ---- the settlement arithmetic (§8.1, §8.4, §2.3), in integer ticks; tolerances V10 / finding F4 ----------------
def value_ticks(p, q):
    # D = q (p_int - O): the trade value V = Q p d in units of K_EUR, an exact integer.
    return int(q) * (int(p) - O_TICKS)


def legs_of(trades):
    # participant -> [(sign, p_int, q_int)]: +1 the seller's leg (receives V), -1 the buyer's (pays V) (§8.4).
    legs = {}
    for _b, _s, b_ag, s_ag, p, q, _sub in trades:
        legs.setdefault(int(s_ag), []).append((+1, int(p), int(q)))
        legs.setdefault(int(b_ag), []).append((-1, int(p), int(q)))
    return legs


def tau_submission(rows):
    # V10's tau_s with c_s = m_s (chunk-agnostic), from the spec's constants. rows = [(p_int, q_int)].
    if not rows:
        return 0.0
    W = sum(int(p) * int(q) for p, q in rows)
    Qs = sum(int(q) for _p, q in rows)
    D = W - O_TICKS * Qs
    m = len(rows)
    return K_EUR * U * (m * W + O_TICKS * Qs + 5 * abs(D)) * (1 + (m + 8) * U)


def tau_participant(legs):
    # Finding F4: tau_a over a participant's m_a legs (unsigned sums).
    m = len(legs)
    W = sum(p * q for _sg, p, q in legs)
    Qa = sum(q for _sg, _p, q in legs)
    G = sum(abs(q * (p - O_TICKS)) for _sg, p, q in legs)
    return K_EUR * U * (m * (W + O_TICKS * Qa) + 5 * G) * (1 + (m + 8) * U)


def ledger(trades):
    # Per row D_i; per submission [rows, Q_s, D_s]; per participant bought / sold / D_a; the gross sum |D_i|.
    D = [value_ticks(t[4], t[5]) for t in trades]
    subs = {}
    for t, d in zip(trades, D):
        e = subs.setdefault(int(t[6]), {"rows": [], "Q": 0, "D": 0})
        e["rows"].append((int(t[4]), int(t[5])))
        e["Q"] += int(t[5])
        e["D"] += d
    legs = legs_of(trades)
    agents = {a: {"bought": sum(q for sg, _p, q in L if sg < 0), "sold": sum(q for sg, _p, q in L if sg > 0),
                  "D": sum(sg * q * (p - O_TICKS) for sg, p, q in L)} for a, L in legs.items()}
    return {"D": D, "gross": sum(abs(d) for d in D), "subs": subs, "legs": legs, "agents": agents}


def readings(legs):
    # The rule's value k D_a for one participant, and the named wrong readings (finding F12), exact from integers.
    right = K_EUR * sum(sg * q * (p - O_TICKS) for sg, p, q in legs)
    return right, {
        WRONG_ALWAYS[0]: 0.0,
        WRONG_ALWAYS[1]: -right,
        WRONG_ALWAYS[2]: right / D_HOURS,                      # Q p x 1 h
        WRONG_ALWAYS[3]: K_EUR * sum(sg * q * p for sg, p, q in legs),
        WRONG_COVER[0]: K_EUR * sum(sg * (wrap32(q * p) - q * O_TICKS) for sg, p, q in legs),
        WRONG_COVER[1]: K_EUR * sum(sg * (wrap32(q * p) - wrap32(q * O_TICKS)) for sg, p, q in legs),
        WRONG_COVER[2]: K_EUR * wrap32(sum(sg * q * (p - O_TICKS) for sg, p, q in legs)),
    }


def discriminated(trades):
    # wrong reading -> True when it misses the rule by more than 5 tau_a for at least one participant.
    out = {n: False for n in WRONG_ALWAYS + WRONG_COVER}
    for _a, L in legs_of(trades).items():
        right, wrong = readings(L)
        tau = tau_participant(L)
        for n, v in wrong.items():
            if abs(v - right) > 5 * tau:
                out[n] = True
    return out


# ---- the construction ---------------------------------------------------------------------------------------------
def _rint(rng, lo_hi):
    lo, hi = lo_hi
    return int(rng.integers(lo, hi + 1))


def _tape(orders):
    tb = synth.TapeBuilder(CONFIG)
    for o in orders:
        tb.add(side=o["side"], price_int=o["price_int"], qty_int=o["qty_int"], agent_id=o["agent"],
               exec_restriction=o["exec"], validity=GFS, expiry_time=0)
    return tb.build()


def _attempt(rng, regime):
    # One attempt of step 2. Returns the orders (with oids) and the attempt's drawn values.
    (lo, hi), q_range = REGIMES[regime]
    mid = O_TICKS + _rint(rng, (lo, hi))
    H = _rint(rng, HALF_WIDTH_ZERO if regime == "zero" else HALF_WIDTH)
    A = _rint(rng, AGENTS)
    book = [(BUY, mid - _rint(rng, (1, H))) for _ in range(_rint(rng, BOOK_PER_SIDE))]
    book += [(SELL, mid + _rint(rng, (1, H))) for _ in range(_rint(rng, BOOK_PER_SIDE))]
    if regime == "zero":
        k = int(rng.integers(0, len(book)))
        book[k] = (book[k][0], mid)                     # a resting order at exactly 0.00 EUR/MWh
    orders = []
    for i in rng.permutation(len(book)):
        side, p = book[int(i)]
        orders.append({"phase": "book", "side": side, "exec": NON, "price_int": int(p),
                       "qty_int": _rint(rng, q_range), "agent": _rint(rng, (1, A))})
    first = [int(x) for x in rng.permutation([BUY, SELL])]
    for j in range(_rint(rng, FLOW)):
        if j < 2:
            side = first[j]
            p = mid + H if side == BUY else mid - H     # through the whole opposite side
        else:
            side = BUY if float(rng.random()) < 0.5 else SELL
            p = mid + _rint(rng, (-H, H))
        ex = IOC if float(rng.random()) < P_IOC else NON
        orders.append({"phase": "flow", "side": side, "exec": ex, "price_int": int(p),
                       "qty_int": _rint(rng, q_range), "agent": _rint(rng, (1, A))})
    pair = None
    if float(rng.random()) < P_SELF_PAIR:
        w = walk(_tape(orders))
        bids = [e["price"] for e in w["book"] if e["side"] == BUY]
        asks = [e["price"] for e in w["book"] if e["side"] == SELL]
        bb, ba = (max(bids) if bids else None), (min(asks) if asks else None)
        mside = BUY if float(rng.random()) < 0.5 else SELL
        if mside == BUY:
            price = bb + 1 if bb is not None else (ba - 1 if ba is not None else mid)
            room = ba is None or price < ba
        else:
            price = ba - 1 if ba is not None else (bb + 1 if bb is not None else mid)
            room = bb is None or price > bb
        if room and 0 <= price <= PRICE_INT_MAX:
            qm = _rint(rng, q_range)
            qi = _rint(rng, (1, qm))
            orders.append({"phase": "self_pair_maker", "side": mside, "exec": NON, "price_int": int(price),
                           "qty_int": qm, "agent": SELF_AGENT})
            orders.append({"phase": "self_pair_ioc", "side": other(mside), "exec": IOC, "price_int": int(price),
                           "qty_int": qi, "agent": SELF_AGENT})
            pair = {"maker_side": mside, "price_int": int(price), "maker_qty_int": qm, "ioc_qty_int": qi}
    for i, o in enumerate(orders):
        o["oid"] = i + 1
    return {"mid_int": int(mid), "half_width_ticks": H, "n_agents": A, "orders": orders, "self_pair": pair}


def _reject_reason(regime, att, w):
    # Step 4, the acceptance test; None = accepted.
    t = w["trades"]
    if {a["side"] for a in w["aggressor"]} != {BUY, SELL}:
        return "a side never aggressed"
    lg = ledger(t)
    if lg["gross"] == 0:
        return "zero gross cash leg"
    disc = discriminated(t)
    miss = [n for n in WRONG_ALWAYS if not disc[n]]
    if miss:
        return "undiscriminated: " + "; ".join(miss)
    if regime == "zero" and not any(r[4] == O_TICKS for r in t):
        return "no trade at exactly 0.00 EUR/MWh"
    if regime == "large_qty" and not any(r[5] * r[4] >= INT32 for r in t):
        return "no row with q * p_int >= 2^31"
    if regime.startswith("extreme") and not any(abs(r[5] * (r[4] - O_TICKS)) >= INT32 for r in t):
        return "no row with |q (p_int - O)| >= 2^31"
    if att["self_pair"] is not None:
        rows9 = [r for r in t if SELF_AGENT in (r[2], r[3])]
        if not rows9 or any(r[2] != r[3] for r in rows9) or lg["agents"][SELF_AGENT]["D"] != 0:
            return "the self-match pair did not trade only with itself"
    return None


def make_draw(seed):
    # One C12 draw (steps 1-4). Raises ConstructionRefused, or synth.CapacityBreach on an analytic-bound breach.
    rng = np.random.Generator(np.random.PCG64(int(seed)))
    regime = REGIME_NAMES[int(rng.choice(len(REGIME_NAMES), p=REGIME_P))]
    rejected = Counter()
    for attempt in range(MAX_TRIES):
        att = _attempt(rng, regime)
        tape = _tape(att["orders"])
        why = _reject_reason(regime, att, walk(tape))
        if why is None:
            break
        rejected[why] += 1
    else:
        raise ConstructionRefused(f"seed {seed} ({regime}): none of {MAX_TRIES} attempts accepted: {dict(rejected)}")
    params = {"seed": int(seed), "regime": regime, "attempt": int(attempt), "rejected_attempts": dict(rejected),
              "mid_int": att["mid_int"], "mid_eur": round(eur(att["mid_int"]), 2),
              "half_width_ticks": att["half_width_ticks"], "n_agents": att["n_agents"],
              "self_pair": att["self_pair"], "orders": [{k: o[k] for k in ORDER_KEYS} for o in att["orders"]]}
    cb = synth.tape_capacity_bounds(tape)
    synth.check_envelope(ENVELOPE, **cb)                                       # CapacityBreach -> refused
    return synth.Draw(construction=CONSTRUCTION, seed=int(seed), tape=tape, params=params, expected={},
                      hop_bound=int(cb["hops"]),
                      capacity_bounds=types.MappingProxyType({a: int(cb[a]) for a in synth.AXES}),
                      oracle_kind="exact-parametric", meta={"built_in": "C12 notebook section 4"})


DRAWN = {}       # seed -> Draw, or None when the builder refused the seed
REFUSED = {}     # seed -> the refusal message


def prepare(seed):
    # Build one seed (the run loop of section 7 calls this for a re-draw too).
    try:
        d = make_draw(seed)
    except (synth.CapacityBreach, ConstructionRefused) as exc:
        d = None
        REFUSED[seed] = f"{type(exc).__name__}: {exc}"
    DRAWN[seed] = d
    return d


for _s in range(SEED0, SEED0 + DRAWS):
    prepare(_s)


def _digest_or_refused(s):
    try:
        return make_draw(s).digest()
    except (synth.CapacityBreach, ConstructionRefused):
        return "refused"


C12_DIGEST = hashlib.blake2b("".join(_digest_or_refused(s) for s in range(8)).encode("ascii"),
                             digest_size=16).hexdigest()


def tape_table(draw, title):
    phase_of = {int(o["oid"]): o["phase"] for o in draw.params["orders"]}
    rows = ["| s | oid | phase | side | exec | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | tape ts |",
            "|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        rows.append(f"| {k} | {int(r[C_OID])} | {phase_of.get(int(r[C_OID]), '?')} | {side_name(r[C_SIDE])} | "
                    f"{EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | {eur(r[C_PRICE]):.2f} | {mw(r[C_QTY]):.1f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = next((s for s in range(SEED0, SEED0 + DRAWS) if DRAWN[s] is not None), None)
if FIRST is None:
    print("every seed of the initial range was refused by the builder")
else:
    _d = DRAWN[FIRST]
    display(Markdown(tape_table(_d, f"{CONSTRUCTION}: seed {FIRST}, regime {_d.params['regime']} "
                                    f"(digest `{_d.digest()[:16]}…`)")))
    print(f"seed {FIRST}: mid {_d.params['mid_eur']:.2f} EUR/MWh (int {_d.params['mid_int']}), half-width "
          f"{_d.params['half_width_ticks']} ticks, {_d.params['n_agents']} participants, self-match pair "
          f"{_d.params['self_pair']}, accepted at attempt {_d.params['attempt']}")
    print(f"seed {FIRST} capacity bounds: {dict(_d.capacity_bounds)}")
print(f"construction digest over seeds 0-7 (construction_digest formula): {C12_DIGEST}")
print(f"seeds refused by the builder in the initial range: {len(REFUSED)}"
      + (f" ({sorted(REFUSED)})" if REFUSED else ""))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
if FIRST is not None:
    _d = DRAWN[FIRST]
    _phase = {int(o["oid"]): o["phase"] for o in _d.params["orders"]}
    ax = axes[0]
    for r in real_rows(_d.tape):
        sgn = 1 if int(r[C_SIDE]) == BUY else -1
        col = "tab:green" if sgn > 0 else "tab:red"
        y = eur(r[C_PRICE])
        if _phase[int(r[C_OID])] == "book":
            ax.barh(y, sgn * mw(r[C_QTY]), height=0.004 * max(1, _d.params["half_width_ticks"]), color=col,
                    edgecolor="black", lw=0.3)
        else:
            ax.plot([sgn * mw(r[C_QTY])], [y], marker=">" if sgn > 0 else "<", color=col, ms=5,
                    mec="black" if int(r[C_EXEC]) == IOC else col)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("MW: buy right, sell left (bars: resting book; markers: order flow, black edge = IOC)", fontsize=7)
    ax.set_ylabel("limit price (EUR/MWh)", fontsize=7)
    ax.set_title(f"seed {FIRST} ({_d.params['regime']}): the book and the flow", fontsize=8)
    lg0 = ledger(walk(_d.tape)["trades"])
    ags = sorted(lg0["agents"])
    vals = [K_EUR * lg0["agents"][a]["D"] for a in ags]
    axes[1].bar([str(a) for a in ags], vals, color=["tab:blue" if v >= 0 else "tab:orange" for v in vals])
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_xlabel("participant (agent id)", fontsize=7)
    axes[1].set_ylabel("expected cash leg k D_a (EUR)", fontsize=7)
    axes[1].set_title(f"seed {FIRST}: expected cash legs (sum {K_EUR * sum(lg0['agents'][a]['D'] for a in ags):+.2f} EUR, "
                      f"gross {K_EUR * lg0['gross']:.2f} EUR)", fontsize=8)
plt.tight_layout()
plt.show()

_ok = [d for d in DRAWN.values() if d is not None]
if _ok:
    _env = {}
    _ords = [o for d in _ok for o in d.params["orders"]]
    _env["rows per draw"] = [len(d.params["orders"]) for d in _ok]
    _env["participants per draw (excl. agent 9)"] = [d.params["n_agents"] for d in _ok]
    _env["mid price (EUR/MWh)"] = [d.params["mid_eur"] for d in _ok]
    _env["half-width (ticks)"] = [d.params["half_width_ticks"] for d in _ok]
    _env["order qty (MW)"] = [mw(o["qty_int"]) for o in _ords]
    _env["order price (EUR/MWh)"] = [eur(o["price_int"]) for o in _ords]
    _env["attempts used per seed"] = [d.params["attempt"] + 1 for d in _ok]
    display(Markdown("**The envelope actually drawn in the initial seed range** (empirical test ranges, never bounds: "
                     "finding F11)\n\n| quantity | min | max |\n|---|---|---|\n"
                     + "\n".join(f"| {k} | {min(v):g} | {max(v):g} |" for k, v in _env.items())))
    print("draws per regime in the initial range:", dict(Counter(d.params["regime"] for d in _ok)))
    print("draws with a self-match pair:", sum(d.params["self_pair"] is not None for d in _ok))


## 5. Known result

**Notation.** Integer ticks throughout. A trade row $i$ carries a price $p_i$ (offset-encoded: $p_i = O$ is 0.00 EUR/MWh, $O = 999\,900$) and a quantity $q_i$, a buyer and a seller (order id and agent), and the index of the submission that wrote it. $k = 0.1 \times 0.01 \times 0.25 = 2.5\times10^{-4}$ EUR per tick·tick, $u = 2^{-24}$.

**The trades: the reference walk (W)** (section 4; §6.1, §6.2 items 1–6, §4.5). Every row `[buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]` is pinned, in submission order and within a submission in walk order. The per-order fates follow: each order's filled quantity; each NON residual still resting at the end; each IOC residual cancelled as `ioc_remainder`; nothing hidden, nothing evicted.

**The value of each trade** (§8.1, §2.3), exactly in integer ticks:

$$V_i = Q_i\,p_i\,d = (0.1\,q_i)\,\big(0.01\,(p_i - O)\big)\,(0.25) = k\,D_i, \qquad D_i = q_i\,(p_i - O) \in \mathbb{Z}.$$

$D_i < 0$ for a trade at a negative price, $D_i = 0$ at exactly 0.00 EUR/MWh. Recomputed from each row **the engine wrote**, $D_i$ must equal the walk's.

**Who pays whom** (§8.4). The seller receives $+V_i$, the buyer pays $-V_i$. Participant $a$'s cash leg is

$$S_a = \sum_{i:\,\text{seller}(i) = a} V_i \;-\; \sum_{i:\,\text{buyer}(i) = a} V_i = k\,D_a, \qquad D_a = \sum_{\text{sells}} D_i - \sum_{\text{buys}} D_i .$$

A self-match row contributes $+D_i - D_i = 0$; the self-match pair's participant has $D_9 = 0$ exactly. (Over any set of rows $\sum_a D_a = 0$ by algebra: shown, never asserted — finding F6.)

**Per submission** $s$ (I6(b)): traded $\iff m_s > 0$; the reported quantity is $Q_s = \sum_{i \in s} q_i$ ticks; the reported settlement is $V_s = k\,D_s$, $D_s = \sum_{i\in s} D_i$ (unsigned: the value of what the submission traded).

**The tolerances** (euro comparisons only).

- Per submission (V10, chunk-agnostic, finding F5): $\tau_s = k\,u\,(m_s W_s + O\,Q_s + 5\lvert D_s\rvert)(1 + (m_s+8)u)$, $W_s = \sum_{i\in s} p_i q_i$. Pinned check: one row $p = 1\,004\,598$, $q = 20$ gives $\tau = 6.043878\times10^{-4}$ EUR.
- Per participant (finding F4): $\tau_a = k\,u\,\big(m_a(W_a + O\,Q_a) + 5\sum_{i}\lvert D_i\rvert\big)(1 + (m_a+8)u)$ over $a$'s $m_a$ legs.

**What the engine must report.** For every participant $a$ on a trade row: $\lvert S^{\text{engine}}_a - k\,D_a\rvert \le \tau_a$, both as recorded by the record builder and as returned by `settle` (the `(M, 6)` call of section 6); for every trading submission: $\lvert V^{\text{engine}}_s - k\,D_s\rvert \le \tau_s$. The engine settles **exactly** the participants on trade rows, from a tape of shape `(M, 6)`, $M$ = the number of trade rows. Non-vacuity: $\sum_i \lvert D_i\rvert > 0$. And I6 (a) and (b) in the battery.

**Discrimination** (finding F12): for each participant, from its legs alone,

| wrong reading | its value |
|---|---|
| zero (vacuous or mis-shaped call) | $0$ |
| sign flipped | $-k D_a$ |
| $d = 1$ h | $4 k D_a$ |
| price not offset-decoded | $k \sum \pm q_i p_i$ |
| int32 wrap of $q\,p_{int}$ | $k \sum \pm (\mathrm{wrap}(q_i p_i) - q_i O)$ |
| int32 wrap of both products | $k \sum \pm (\mathrm{wrap}(q_i p_i) - \mathrm{wrap}(q_i O))$ |
| int32 wrap of $D_a$ | $k\,\mathrm{wrap}(D_a)$ |

A reading is discriminated on a draw if, for some participant, it misses $k D_a$ by more than $5\tau_a$ (its own rounding is then no excuse). The first four must be discriminated on every valid draw; each wrap reading on at least one.

**What is asserted, per valid draw** (expected and observed recorded in full as canonical compact JSON):

| key | expected | read from the run |
|---|---|---|
| `accepted`, `rejected` | every order id; none | `accepted`, `rejections` |
| `trades` | (W)'s rows, in order | `fills`, by (submission, row) |
| `trade_value_ticks` | $D_i$ per row | $q\,(p - O)$ of each **observed** row |
| `submissions` | `[traded, Q_s, D_s]` for every submission | `engine_reports`: `traded`, `round(trade_qty_mw / 0.1)`; $D_s$ from the observed rows |
| `ledger` | `[bought, sold, D_a]` per participant | the observed rows, by buyer and seller agent |
| `gross_value_ticks` | $\sum_i\lvert D_i\rvert$ | the observed rows |
| `settled_agents` | the participants on trade rows | the keys of `engine_settlements` |
| `settlement_tape_shape` | `[M, 6]` | the shape `settle` passed |
| `filled`, `still_resting`, `cancelled`, `hidden_remaining`, `evicted` | (W)'s fates | the record's per-order maps |
| *(euro assertion)* `agents`, `submissions` | `[k D, tau]` | `engine_settlements[a]`, `settle(rec)[1][a]`; `settlement_value_eur` |

Separately (section 7): the dataset precondition, the configuration constants and the tolerance pin, the discrimination, the gross cash leg, the settlement-tape shape, the battery (RB, I2, I3, I6, I7, I9 pass; I1, I4, I5 never fail), the capacity checks, coverage and the eager comparisons.

The code below derives everything from the tape through (W). It never reads `draw.expected`, and it never runs the engine.


In [ ]:
def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def _maps_from_walk(oids, w):
    filled = {o: 0 for o in oids}
    for t in w["trades"]:
        filled[t[0]] += t[5]
        filled[t[1]] += t[5]
    return ({str(o): filled[o] for o in oids if filled[o]},
            {str(e["oid"]): e["qty"] for e in sorted(w["book"], key=lambda e: e["oid"])},
            {str(o): [["ioc_remainder", q]] for o, q in sorted(w["ioc_cancelled"].items())})


def expectation(draw):
    # Everything section 7 compares for one draw, from (W) over the tape's rows and the settlement arithmetic.
    rows = real_rows(draw.tape)
    oids = [int(r[C_OID]) for r in rows]
    w = walk(draw.tape)
    t = w["trades"]
    lg = ledger(t)
    filled, resting, cancelled = _maps_from_walk(oids, w)
    exact = {
        "accepted": sorted(oids), "rejected": [],
        "trades": t,
        "trade_value_ticks": lg["D"],
        "submissions": {str(s): [s in lg["subs"], lg["subs"][s]["Q"] if s in lg["subs"] else 0,
                                 lg["subs"][s]["D"] if s in lg["subs"] else 0] for s in range(len(rows))},
        "ledger": {str(a): [v["bought"], v["sold"], v["D"]] for a, v in sorted(lg["agents"].items())},
        "gross_value_ticks": lg["gross"],
        "settled_agents": sorted(lg["agents"]),
        "settlement_tape_shape": [len(t), 6],
        "filled": filled, "still_resting": resting, "cancelled": cancelled,
        "hidden_remaining": {}, "evicted": {},
    }
    euro = {"agents": {str(a): [K_EUR * v["D"], tau_participant(lg["legs"][a])] for a, v in sorted(lg["agents"].items())},
            "submissions": {str(s): [K_EUR * e["D"], tau_submission(e["rows"])] for s, e in sorted(lg["subs"].items())}}
    disc = discriminated(t)
    p = draw.params
    classes = [f"regime: {p['regime']}"]
    if p.get("self_pair") is not None:
        classes.append(SELF_PAIR_CLASS)
    tc = TRADE_CLASSES
    if any(r[4] < O_TICKS for r in t):
        classes.append(tc[0])
    if any(r[4] == O_TICKS for r in t):
        classes.append(tc[1])
    if any(r[2] == r[3] for r in t):
        classes.append(tc[2])
    if any(v["bought"] > 0 and v["sold"] > 0 for v in lg["agents"].values()):
        classes.append(tc[3])
    if any(len(e["rows"]) >= 2 for e in lg["subs"].values()):
        classes.append(tc[4])
    if any(a["exec"] == IOC for a in w["aggressor"]):
        classes.append(tc[5])
    if w["rested_after_trade"]:
        classes.append(tc[6])
    if any(r[5] * r[4] >= INT32 for r in t):
        classes.append(tc[7])
    if any(abs(r[5] * (r[4] - O_TICKS)) >= INT32 for r in t):
        classes.append(tc[8])
    if any(abs(v["D"]) >= INT32 for v in lg["agents"].values()):
        classes.append(tc[9])
    classes += [f"discriminated: {n}" for n in WRONG_COVER if disc[n]]
    return {"exact": exact, "euro": euro, "classes": classes,
            "draw_classes": [c for c in classes if c in EAGER_CLASSES], "disc": disc,
            "zero_sum_ticks": sum(v["D"] for v in lg["agents"].values()),   # algebra: shown, never asserted (F6)
            "gross_eur": K_EUR * lg["gross"], "expects_trade": True}


# ---- the dataset precondition (section 4) ------------------------------------------------------------------------
def precondition(draw):
    # Returns the list of departures (empty: the draw is as section 4 describes).
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    if len(rows) < 4:
        return [f"{len(rows)} real rows"]
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids, ts = [int(x[C_OID]) for x in rows], [int(x[C_TS]) for x in rows]
    if oids != list(range(1, len(rows) + 1)) or any(b <= a for a, b in zip(ts, ts[1:])):
        bad.append(f"order ids {oids} / tape timestamps {ts} not 1..N / strictly increasing")
    for x in rows:
        o = int(x[C_OID])
        if int(x[C_EXEC]) not in (NON, IOC):
            bad.append(f"oid {o}: exec restriction {int(x[C_EXEC])} (only NON and IOC are drawn)")
        if int(x[C_VALIDITY]) != GFS or int(x[C_EXPIRY]) != 0 or int(x[C_PEAK]) != 0 or int(x[C_PPD]) != 0:
            bad.append(f"oid {o}: not a plain GFS row (validity, expiry, peak, increment)")
        if not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX:
            bad.append(f"oid {o}: price or quantity outside §2.3")
    try:
        regime, mid, H, A = p["regime"], int(p["mid_int"]), int(p["half_width_ticks"]), int(p["n_agents"])
        ords = p["orders"]
        p_rows = [[int(q["oid"]), int(q["side"]), int(q["exec"]), int(q["price_int"]), int(q["qty_int"]), int(q["agent"])]
                  for q in ords]
        t_rows = [[int(x[C_OID]), int(x[C_SIDE]), int(x[C_EXEC]), int(x[C_PRICE]), int(x[C_QTY]), int(x[C_AGENT])]
                  for x in rows]
        if p_rows != t_rows:
            bad.append("params != tape")
        (lo, hi), (qlo, qhi) = REGIMES[regime]
        if not lo <= mid - O_TICKS <= hi:
            bad.append(f"mid {mid} outside the {regime} regime")
        hw = HALF_WIDTH_ZERO if regime == "zero" else HALF_WIDTH
        if not hw[0] <= H <= hw[1] or not AGENTS[0] <= A <= AGENTS[1]:
            bad.append(f"half-width {H} or participant count {A} outside the envelope")
        phases = [q["phase"] for q in ords]
        nb = phases.count("book")
        if phases[:nb] != ["book"] * nb or any(ph not in ("book", "flow", "self_pair_maker", "self_pair_ioc")
                                                for ph in phases):
            bad.append(f"phases out of order: {phases}")
        book = ords[:nb]
        bids = [int(q["price_int"]) for q in book if int(q["side"]) == BUY]
        asks = [int(q["price_int"]) for q in book if int(q["side"]) == SELL]
        if not bids or not asks or max(bids) >= min(asks) or any(int(q["exec"]) != NON for q in book):
            bad.append("the resting book is one-sided, crossed, or not all NON")
        if any(not -H <= int(q["price_int"]) - mid <= H for q in ords if q["phase"] in ("book", "flow")):
            bad.append("a book or flow price outside mid +- half-width")
        if any(not qlo <= int(q["qty_int"]) <= qhi for q in ords if q["phase"] != "self_pair_ioc"):
            bad.append("a quantity outside the regime's range")   # the pair's IOC: 1..maker qty, checked below
        flow = [q for q in ords if q["phase"] == "flow"]
        if len(flow) < 2 or sorted((int(q["side"]), int(q["price_int"])) for q in flow[:2]) != \
                sorted([(BUY, mid + H), (SELL, mid - H)]):
            bad.append("the first two flow rows are not one BUY at mid + H and one SELL at mid - H")
        if any(not 1 <= int(q["agent"]) <= A for q in ords if q["phase"] in ("book", "flow")):
            bad.append("a book or flow agent outside 1..A")
        pair = [q for q in ords if q["phase"].startswith("self_pair")]
        if (p.get("self_pair") is None) != (not pair):
            bad.append("self_pair params disagree with the rows")
        if pair:
            mk, ioc = pair if len(pair) == 2 else (None, None)
            if (mk is None or [mk["phase"], ioc["phase"]] != ["self_pair_maker", "self_pair_ioc"]
                    or ords[-2:] != pair or int(mk["agent"]) != SELF_AGENT or int(ioc["agent"]) != SELF_AGENT
                    or int(mk["exec"]) != NON or int(ioc["exec"]) != IOC or int(ioc["side"]) != other(mk["side"])
                    or int(ioc["price_int"]) != int(mk["price_int"]) or int(ioc["qty_int"]) > int(mk["qty_int"])):
                bad.append("the self-match pair is not the last two rows as section 4 describes")
    except Exception as exc:  # noqa: BLE001
        bad.append(f"params unreadable: {type(exc).__name__}: {exc}")
    if _reject_reason(p.get("regime", "?"), {"self_pair": p.get("self_pair")}, walk(t)) is not None:
        bad.append("the draw does not pass the acceptance test of section 4")
    return bad


# ---- the expectation on the first draw, and the classes of the initial range ------------------------------------
if FIRST is not None:
    _d = DRAWN[FIRST]
    _e = expectation(_d)
    _x = _e["exact"]
    print(f"seed {FIRST} classes: {_e['classes']}")
    print(f"  trades [buyer_oid, seller_oid, buyer_agent, seller_agent, price_int, qty_int, submission]: {_x['trades']}")
    print(f"    prices in EUR/MWh: {[round(eur(t[4]), 2) for t in _x['trades']]}")
    print(f"  D_i = q (p - O): {_x['trade_value_ticks']}  -> V_i (EUR, reported): "
          f"{[round(K_EUR * d, 4) for d in _x['trade_value_ticks']]}")
    print(f"  ledger [bought, sold, D_a]: {_x['ledger']}")
    print(f"  expected cash legs k D_a and tau_a (EUR): { {a: [round(v, 4), float(f'{t:.3e}')] for a, (v, t) in _e['euro']['agents'].items()} }")
    print(f"  gross |V| = {_e['gross_eur']:.4f} EUR; sum_a D_a = {_e['zero_sum_ticks']} (algebra, not a check)")
    print(f"  discriminated wrong readings: { {n: v for n, v in _e['disc'].items()} }")
    print(f"  precondition departures: {precondition(_d)}")
    _pt = tau_submission([TAU_PIN[:2]])
    print(f"V10 pin: tau(p={TAU_PIN[0]}, q={TAU_PIN[1]}) = {_pt:.7e} EUR (pinned {TAU_PIN[2]:.7e})")
_init_classes = Counter(c for d in DRAWN.values() if d is not None for c in expectation(d)["classes"])
display(Markdown("**Classes in the initial seed range** (draws containing each)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {c} | {_init_classes.get(c, 0)} |" for c in ALL_CLASSES)))


## 6. Run

### Eager and jitted, and the flattened settlement call (stub — to be wired at K3)

> **K2 stub.** This section is deliberately empty. The developer wires it at K3 and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation.

Section 7 consumes exactly **two names** from this section. While either is not defined, section 7 records a failing `section 6 wired` check, runs no draw, and the notebook ends `NOT CERTIFIED`.

**`run_draw(draw, *, jit) -> RunRecord`**: the battery's record builder on one draw, i.e. `tests.omiecid.reference.omie_invariants.record_from_draw(draw, jit=jit)`, with **no other argument** (no gate expiry, no cancellations). It runs `draw.tape` **as is** (nothing in it may be rewritten) on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`) and the draw's own analytic capacity bounds (`draw.capacity_bounds`). `draw` is a `Draw` built by section 4's `make_draw(seed)`. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. Every call starts from its own fresh book. It may raise; section 7 records a raise as a failed check.

**`settle(rec) -> (shape, settlements)`**: this notebook's own settlement call (card: *"always flattens the trades to (M, 6) before settling, as the production callers do, asserts the shape it passes"*). From `rec.fills` sorted by `(submission, row_index)`, build a numpy **int32** array `T` of shape `(M, 6)`, $M$ = `len(rec.fills)`, one row per fill, columns `[sec, price_int, qty_int, buyer_agent, seller_agent, contract_id]` (`contract_id` = the tape's contract, 0; nothing else reshaped, no `(steps, 2, 6)` tape anywhere). Then, for every agent that appears as a buyer or seller in a fill, call `omiecid_simulator.omiecid.settlement.compute_agent_pnl(jnp.asarray(T), jnp.int32(a))` **eagerly and plainly** — not under `jax.jit`, and without an outer `jax.experimental.enable_x64()` context, as the production callers do (OQ-18) — and convert the result with `float(...)`. Return `(tuple(T.shape), {a: value})` (the shape **actually passed**). With no fills: `((0, 6), {})`. It may raise; section 7 records a raise as a failed check.

Section 7 reads these `RunRecord` fields:
- `n_submissions`, `accepted` (its keys) and `rejections` (each `Rejection.oid`);
- `fills`: each `Fill`'s `submission`, `row_index`, `buyer_oid`, `seller_oid`, `buyer_agent`, `seller_agent`, `price_int`, `qty_int`;
- `engine_reports`: each `EngineReport`'s `submission`, `traded`, `trade_qty_mw`, `settlement_value_eur`;
- `engine_settlements` (agent id → EUR);
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.cause` and `Cancellation.qty_int`), per order id;
- `observed_capacity`;
- the whole record, through `run_battery` (each check's `status`, I6's and I8's `evidence`) and `compare_records`.

Section 7 calls `run_draw(..., jit=True)` and then `settle(...)` on every draw it uses, and `run_draw(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is built with `prepare(seed)` before the call.


In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw(draw, *, jit) -> RunRecord and settle(rec) -> (shape, settlements),
# per the contract above.
#
# run_draw: the battery's record builder on one draw with no other argument.
# record_from_draw runs draw.tape as is on one fresh book of one OPEN contract, with the
# default configs (OMIEProductConfig(), OMIEMarketConfig()) and the draw's own analytic
# capacity bounds (draw.capacity_bounds); no C12 draw schedules a cancel and no gate expiry
# is applied. jit=False wraps the whole run in jax.disable_jit() (inside build_record);
# jit=True runs it jitted. Every call starts from its own fresh book. It may raise.
#
# settle: the flattened (M, 6) int32 settlement tape built from rec.fills, sorted by
# (submission, row_index), columns [sec, price_int, qty_int, buyer_agent, seller_agent,
# contract_id]; then compute_agent_pnl called eagerly and plainly (no jax.jit, no outer
# enable_x64: OQ-18's eager contract) once per trade-row agent. It may raise.
# ---------------------------------------------------------------------------------------
import jax.numpy as jnp

from omiecid_simulator.omiecid.settlement import compute_agent_pnl
from tests.omiecid.reference.omie_invariants import record_from_draw

SETTLE_CONTRACT_ID = 0   # the tape's single contract (section 4: every row on contract 0)


def run_draw(draw, *, jit):
    """One run of `draw.tape` (unmodified) on a fresh OPEN book: the battery's record builder."""
    return record_from_draw(draw, jit=jit)


def settle(rec):
    """This notebook's own (M, 6) settlement call over `rec.fills`; returns (shape passed, {agent: EUR})."""
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    if not fills:
        return (0, 6), {}
    T = np.asarray([[int(f.sec), int(f.price_int), int(f.qty_int), int(f.buyer_agent), int(f.seller_agent),
                     SETTLE_CONTRACT_ID] for f in fills], dtype=np.int32)
    assert T.ndim == 2 and T.shape == (len(fills), 6), T.shape   # the shape hazard (F8)
    agents = sorted({int(f.buyer_agent) for f in fills} | {int(f.seller_agent) for f in fills})
    tape = jnp.asarray(T)
    return tuple(T.shape), {a: float(compute_agent_pnl(tape, jnp.int32(a))) for a in agents}


print("run_draw: record_from_draw(draw, jit=jit) on one OPEN contract, default configs, the draw's capacity bounds, "
      "no cancels, no GCT expiry; settle: (M, 6) int32 tape from the sorted fills, compute_agent_pnl eager per "
      "trade-row agent")

## 7. Certification

The oracle comparison comes first, one **integer** assertion and one **euro-within-tolerance** assertion per valid draw; then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw.** (1) One assertion named by seed and its eager classes, whose `expected` and `observed` are the canonical compact JSON (sorted keys) of section 5's integer table, **in full** and never truncated; its `detail` lists every key that differs. (2) One assertion whose `expected` is `{agents: {a: [k D_a, tau_a]}, submissions: {s: [k D_s, tau_s]}}` and whose `observed` is `{agents: {a: [engine_settlements[a], settle(rec)[1][a]]}, submissions: {s: settlement_value_eur}}`, in full; it passes when every observed value lies within its tolerance, and its `detail` lists every miss and reports the largest error over its tolerance. The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; `OMIEProductConfig()` carries the specification's ticks, price floor and $d$ (finding F3); the tolerance formula reproduces V10's pinned example; the dataset precondition (section 4), listing every departing seed and its departures; the **discrimination** of the four always-readings on every valid draw, and coverage of the three int32-wrap readings (finding F12); the **gross cash leg** $\sum\lvert D_i\rvert > 0$ from the observed rows on every valid draw (I9, V3-03); the **settlement tape** passed as `(M, 6)` on every valid draw (F8); the battery — **RB, I2, I3, I6, I7 and I9 pass** on every valid draw, and **I1, I4 and I5 never fail** (not-applicable counts reported, finding F9); **no capacity flag inside the margin** (finding F10); the **hop bound** ≤ 160 on every valid draw; **no iceberg row** (the slice ceiling holds trivially); **coverage** — every class of section 3 among the valid draws; the eager comparisons — at least one in all and one per eager class, a missing one recorded as a failure, and each eager record reconciled within the tolerance.

The battery runs on every jitted run with `expects_trade=True` (every draw implies trades). **Reported, not asserted:** the valid draws per class, the discards per reason, the eager seeds, the battery status counts, the largest error over tolerance (participants and submissions), the I6(a) sum of the engine's settlements against $\sum_s\tau_s$, and the gross cash leg in euros.


In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Settlement and the cash leg",
    known_result="Every trade row pinned by the walk (counterparties included); each trade's value D_i = q (p_int - O) "
                 "recomputed exactly in ticks from its own row (V = Q p d, d = 0.25 h, k = 2.5e-4 EUR per tick-tick); "
                 "each participant's engine settlement within tau_a of k D_a, D_a = sells - buys (§8.4 sign convention), "
                 "from an (M, 6) settlement tape, for exactly the participants on trade rows; each trading submission's "
                 "reported settlement within tau_s of k D_s and its reported quantity exact; gross sum |D_i| > 0; I6(a) "
                 "and (b) pass; the integer zero-sum is algebra and not asserted",
    oracle_kind="exact-parametric, carried by I6", quotes=QUOTES, citations=CITATIONS, constructions=(),
    requested_draws=DRAWS, runtime_forecast_s=180.0, started_at=T0,
    downstream="Track B scoring (every euro reward is a settlement), C15 (I6 at scale), C16 (d as a contract "
               "property), and every scenario whose battery runs I6 (C04-C11, C13-C16); the certificate's §8.3 "
               "per-unit gap stays open",
)

R_VALUE = "§8.1 V = Q x p x d, d = 0.25 h; §2.3 ticks and price floor; §11.1 row 'Trade value = Q x p x d'"
R_SIGN = "§8.4 (buyer -V, seller +V, net = sells - buys; energy value exogenous: cash leg only)"
R_WALK = "§6.1; §6.2 items 1-6; §4.5 NON/IOC rows; §5.1; review D3-08 (counterparties pinned)"
R_VAC = "review V3-03 (gross sum |V| > 0; per-participant recomputation); I9"
R_DRAW = "§8.1; §8.4; §2.3; §6.1-§6.2; §6.6; D3-08"   # short: repeated on every draw
R_EURO = "I6 as restated (V10 tau_s, chunk-agnostic); finding F4 tau_a; §8.1; §8.4; OQ-18 (eager settlement)"
R_SHAPE = "OMIE-AGENT-PNL-SHAPE-GUARD-1 hazard (card): the settlement tape is (M, 6)"
R_BAT = "the battery (RB, I1-I9) on every valid draw"
R_I6 = "I6 (a) and (b) as restated in the battery (V10)"
R_CAP = "capacity-validity precondition (five ceilings)"
R_DATA = "dataset precondition (section 4); §2.3"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
R_DISC = "rule 1 (every check seen failing): each named wrong settlement misses the rule by > 5 tau_a"
R_CONST = "finding F3: the expectation's constants are the specification's (§2.3, §8.1)"

MUST_PASS = ("RB", "I2", "I3", "I6", "I7", "I9")   # applicable to every C12 draw: status 'pass'
NO_FAIL = ("I1", "I4", "I5")                        # not_applicable only on a run with no fills: never 'fail' (F9)


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a value", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_DRAW)
        return None


def nonzero(m):
    return {str(int(k)): int(v) for k, v in sorted(m.items(), key=lambda kv: int(kv[0])) if int(v)}


def observed_trades(rec):
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    return [[int(f.buyer_oid), int(f.seller_oid), int(f.buyer_agent), int(f.seller_agent), int(f.price_int),
             int(f.qty_int), int(f.submission)] for f in fills]


def observe(rec, settle_out):
    # The observed counterpart of section 5's integer table (the same per-row arithmetic, on the rows the engine wrote).
    t = observed_trades(rec)
    lg = ledger(t)
    reports = {int(er.submission): er for er in rec.engine_reports}
    subs = {}
    for s in range(int(rec.n_submissions)):
        er = reports.get(s)
        subs[str(s)] = None if er is None else [bool(er.traded), int(round(float(er.trade_qty_mw) / QTY_TICK_MW)),
                                                lg["subs"][s]["D"] if s in lg["subs"] else 0]
    return {
        "accepted": sorted(int(a) for a in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": t,
        "trade_value_ticks": lg["D"],
        "submissions": subs,
        "ledger": {str(a): [v["bought"], v["sold"], v["D"]] for a, v in sorted(lg["agents"].items())},
        "gross_value_ticks": lg["gross"],
        "settled_agents": sorted(int(a) for a in rec.engine_settlements),
        "settlement_tape_shape": None if settle_out is None else [int(x) for x in settle_out[0]],
        "filled": nonzero(rec.filled),
        "still_resting": nonzero(rec.still_resting),
        "cancelled": {str(int(q)): [[str(c.cause), int(c.qty_int)] for c in cs]
                      for q, cs in sorted(rec.cancelled_or_expired.items(), key=lambda kv: int(kv[0])) if cs},
        "hidden_remaining": nonzero(rec.hidden_remaining),
        "evicted": nonzero(rec.evicted),
    }


RATIO = {"agents": 0.0, "submissions": 0.0}   # largest |observed - expected| / tau (reported only)


def check_euro(seed, e, rec, settle_map, label):
    # The euro assertion: every settlement and report within its derived tolerance. Returns ok.
    x = e["euro"]
    reports = {int(er.submission): er for er in rec.engine_reports}
    obs_a, obs_s, miss = {}, {}, []
    for a, (v, tau) in x["agents"].items():
        got = [rec.engine_settlements.get(int(a))]
        if settle_map is not None:
            got.append(settle_map.get(int(a)))
        obs_a[a] = [None if g is None else float(g) for g in got]
        for src, g in zip(("record", "settle"), obs_a[a]):
            if g is None or not abs(g - v) <= tau:
                miss.append(f"participant {a} ({src}): {g!r} vs {v!r} +- {tau:.3e}")
            elif tau > 0:
                RATIO["agents"] = max(RATIO["agents"], abs(g - v) / tau)
    for s, (v, tau) in x["submissions"].items():
        er = reports.get(int(s))
        g = None if er is None else float(er.settlement_value_eur)
        obs_s[s] = g
        if g is None or not abs(g - v) <= tau:
            miss.append(f"submission {s}: {g!r} vs {v!r} +- {tau:.3e}")
        elif tau > 0:
            RATIO["submissions"] = max(RATIO["submissions"], abs(g - v) / tau)
    ok = not miss
    return cert.check(f"seed {seed}{label}: settlements and reports within the derived tolerance", ok, seed=seed,
                      expected=canon(x), observed=canon({"agents": obs_a, "submissions": obs_s}), unit="other",
                      detail=("misses: " + "; ".join(miss)) if miss else
                      f"all within; largest err/tau so far: participants {RATIO['agents']:.3g}, "
                      f"submissions {RATIO['submissions']:.3g}", rule=R_EURO)


def check_draw(seed, e, rec, settle_out):
    # Section 5's integer table for one draw, as one assertion with expected / observed in full. Returns ok.
    o = observe(rec, settle_out)
    x = e["exact"]
    diff = [k for k, v in x.items() if o.get(k) != v]
    return cert.check(f"seed {seed} [{', '.join(e['draw_classes'])}]", not diff, seed=seed,
                      expected=canon(x), observed=canon(o), unit="other",
                      detail=("differs on: " + ", ".join(diff)) if diff else "identical", rule=R_DRAW)


EAGER_DONE = defaultdict(list)   # "any valid draw" or eager class -> seeds with an eager comparison


def eager_compare(seed, draw, rec, e, tags):
    # Run the draw eager: compare its integers with the jitted record, and reconcile its euros.
    rec_e = guarded(f"seed {seed} (eager)", seed, run_draw, draw, jit=False)
    diff = ["eager run missing"] if rec_e is None else list(compare_records(rec_e, rec))
    for t in ["any valid draw"] + list(tags):
        EAGER_DONE[t].append(seed)
    ok = not diff
    cert.check(f"eager == jitted (seed {seed})", ok, kind="eager_jit", seed=seed, expected=True, observed=ok,
               unit="bool", detail="identical" if ok else ", ".join(diff), rule=R_EAGER)
    if rec_e is not None:
        check_euro(seed, e, rec_e, None, " (eager)")


def margin_breaches(draw):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = []
    for name, bd in (("draw", dict(draw.capacity_bounds)), ("generic", synth.tape_capacity_bounds(draw.tape))):
        bad += [f"{name} {k} bound {bd.get(k)} > {m}" for k, m in lim.items() if int(bd.get(k, 0)) > m]
    if any(int(r[C_EXEC]) == ICE for r in real_rows(draw.tape)):
        bad.append("an iceberg row (none is drawn)")
    return bad


# ---- constants and the tolerance pin (no run needed) ------------------------------------------------------------
_cfg = [CONFIG.price_tick, CONFIG.qty_tick, CONFIG.price_min, CONFIG.delivery_duration_hours]
_spec = [PRICE_TICK_EUR, QTY_TICK_MW, PRICE_FLOOR_EUR, D_HOURS]
cert.check("OMIEProductConfig() carries the specification's price tick, qty tick, price floor and d (F3)",
           _cfg == _spec, expected=_spec, observed=_cfg, unit="other", rule=R_CONST)
_pin = tau_submission([TAU_PIN[:2]])
cert.check("tau_s reproduces V10's pinned example (p = 1,004,598, q = 20 -> 6.043878e-4 EUR, 7 significant digits)",
           f"{_pin:.6e}" == f"{TAU_PIN[2]:.6e}", expected=f"{TAU_PIN[2]:.6e}", observed=f"{_pin:.6e}", unit="other",
           detail=f"unrounded {_pin!r}", rule=R_I6)
wired = callable(globals().get("run_draw")) and callable(globals().get("settle"))
cert.check("section 6 wired (run_draw and settle defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_DRAW)

PER_DRAW, DATA_BAD, NO_DISCRIM, GROSS_BAD, SHAPE_BAD = [], {}, {}, [], {}
VALID = []
COVER = Counter()
FLAG_INSIDE_MARGIN = {}                    # seed -> I8 evidence, for draws that passed the pre-run margin
STATUS = defaultdict(Counter)              # check id -> status counts over valid draws
BAT_BAD = defaultdict(list)                # check id -> seeds with a status that is not allowed
DISCARDS = Counter()
PEAK = {"hop_bound": 0, "generic_hop_bound": 0, "trades": 0}
I6_SUM = {"max_abs_sum_eur": 0.0, "max_sum_over_tau_sum": 0.0}
GROSS_EUR = []


def run_all():
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + MAX_ATTEMPTS:
        draw = DRAWN[seed] if seed in DRAWN else prepare(seed)
        if draw is None:
            cert.discard(seed, f"builder refused the seed: {REFUSED.get(seed)}")
            DISCARDS["builder refused"] += 1
            seed += 1
            continue
        bad = precondition(draw)
        if bad:
            DATA_BAD[str(seed)] = bad
            cert.check(f"seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[], observed=bad,
                       unit="other", rule=R_DATA)
            seed += 1
            continue
        mb = margin_breaches(draw)
        if mb:
            cert.discard(seed, "pre-run capacity margin: " + canon(mb))
            DISCARDS["pre-run margin"] += 1
            seed += 1
            continue
        e = expectation(draw)
        rec = guarded(f"seed {seed} (jit)", seed, run_draw, draw, jit=True)
        if rec is None:
            seed += 1
            continue
        so = guarded(f"seed {seed} (settle)", seed, settle, rec)
        rep = run_battery(rec, expects_trade=e["expects_trade"])
        i8 = next(r for r in rep.results if r.check_id == "I8")
        if not cert.add_battery(rep, seed=seed):
            FLAG_INSIDE_MARGIN[str(seed)] = dict(i8.evidence)
            DISCARDS["I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F10)
            continue
        valid += 1
        VALID.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION})
        for r in rep.results:
            STATUS[r.check_id][r.status] += 1
            if (r.check_id in MUST_PASS and r.status != "pass") or (r.check_id in NO_FAIL and r.status == "fail"):
                BAT_BAD[r.check_id].append(seed)
            if r.check_id == "I6" and r.evidence:
                ssum, tsum = float(r.evidence.get("settlement_sum", 0.0)), float(r.evidence.get("tau_sum", 0.0))
                I6_SUM["max_abs_sum_eur"] = max(I6_SUM["max_abs_sum_eur"], abs(ssum))
                if tsum > 0:
                    I6_SUM["max_sum_over_tau_sum"] = max(I6_SUM["max_sum_over_tau_sum"], abs(ssum) / tsum)
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(draw.tape)["hops"]))
        PEAK["trades"] = max(PEAK["trades"], len(rec.fills))
        for c in e["classes"]:
            COVER[c] += 1
        und = [n for n in WRONG_ALWAYS if not e["disc"][n]]
        if und:
            NO_DISCRIM[str(seed)] = und
        g = ledger(observed_trades(rec))["gross"]
        GROSS_EUR.append(K_EUR * g)
        if not g > 0:
            GROSS_BAD.append(seed)
        shape = None if so is None else [int(x) for x in so[0]]
        if shape != [len(rec.fills), 6]:
            SHAPE_BAD[str(seed)] = {"passed": shape, "fills": len(rec.fills)}
        ok = check_draw(seed, e, rec, so)
        ok_e = check_euro(seed, e, rec, None if so is None else so[1], "")
        PER_DRAW.append({"seed": seed, "classes": e["draw_classes"], "ok": ok and ok_e})
        if valid <= EAGER_DRAWS or any(c not in EAGER_DONE for c in e["draw_classes"]):
            eager_compare(seed, draw, rec, e, e["draw_classes"])
        seed += 1
    return valid


N_VALID = run_all() if wired else 0

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{N_VALID} valid draws", rule=R_DATA)
cert.check("discrimination: zero, sign-flipped, d = 1 h and undecoded-price settlements each miss the rule on every "
           "valid draw (F12)", wired and N_VALID > 0 and not NO_DISCRIM, expected={}, observed=NO_DISCRIM,
           unit="other", detail=f"valid draws: {N_VALID}", rule=R_DISC)
cert.check("gross cash leg sum |D_i| > 0 from the observed rows on every valid draw (I9, V3-03)",
           wired and N_VALID > 0 and not GROSS_BAD, expected=[], observed=GROSS_BAD, unit="other",
           detail=f"gross cash leg over valid draws: min {min(GROSS_EUR, default=0):.4g} EUR, "
                  f"max {max(GROSS_EUR, default=0):.4g} EUR (reported)", rule=R_VAC)
cert.check("settle passed an (M, 6) settlement tape, M = the number of fills, on every valid draw (F8)",
           wired and N_VALID > 0 and not SHAPE_BAD, expected={}, observed=SHAPE_BAD, unit="other", rule=R_SHAPE)
for _cid in MUST_PASS:
    cert.check(f"{_cid} passes on every valid draw", wired and N_VALID > 0 and not BAT_BAD[_cid],
               expected=[], observed=BAT_BAD[_cid], unit="other",
               detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_cid])}",
               rule=R_I6 if _cid == "I6" else R_BAT)
for _cid in NO_FAIL:
    cert.check(f"{_cid} never fails on a valid draw (not_applicable allowed: a run with no fills)",
               wired and N_VALID > 0 and not BAT_BAD[_cid], expected=[], observed=BAT_BAD[_cid], unit="other",
               detail=f"status counts over {N_VALID} valid draws: {dict(STATUS[_cid])}; "
                      f"not_applicable on {STATUS[_cid]['not_applicable']} draws (reported, not asserted)",
               rule=R_BAT)
cert.check("no capacity flag (truncated, book_full, saturated ring) on a draw inside the analytic margin (F10)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check("hop bound <= 160 on every valid draw (largest reported: draw bound, generic bound)",
           wired and N_VALID > 0 and PEAK["hop_bound"] <= ENVELOPE.max_hops
           and PEAK["generic_hop_bound"] <= ENVELOPE.max_hops,
           expected=f"<= {ENVELOPE.max_hops}", observed=[PEAK["hop_bound"], PEAK["generic_hop_bound"]],
           unit="count", rule=R_CAP)
cert.check("trades <= 80 on every valid draw (largest observed)", wired and N_VALID > 0
           and PEAK["trades"] <= ENVELOPE.max_trades, expected=f"<= {ENVELOPE.max_trades}", observed=PEAK["trades"],
           unit="count", rule=R_CAP)
_got = {c: COVER[c] for c in ALL_CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every class (section 3), the three int32-wrap readings included",
           wired and not _missing, expected={c: ">= 1" for c in ALL_CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- eager: at least one comparison, and one per eager class (absence = failure) -------------------------------
for _t in ("any valid draw",) + EAGER_CLASSES:
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_DRAW)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws: {N_VALID}; per class: {dict(COVER)}")
print(f"discards per reason: {dict(DISCARDS)}")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"battery status counts over valid draws: { {k: dict(v) for k, v in STATUS.items()} }")
print(f"largest err/tau: participants {RATIO['agents']:.3g}, submissions {RATIO['submissions']:.3g} (reported)")
print(f"I6(a): largest |sum of engine settlements| {I6_SUM['max_abs_sum_eur']:.4g} EUR; largest ratio to sum tau_s "
      f"{I6_SUM['max_sum_over_tau_sum']:.3g} (reported)")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']} (generic {PEAK['generic_hop_bound']}); "
      f"largest observed trade count {PEAK['trades']}")
display(Markdown(cert.table()))


## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.


In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")


In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))
